In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

In [4]:
if torch.cuda.is_available():
    # Print the name of the GPU
    print(f"GPU: {torch.cuda.get_device_name(0)} is available.")
    # Print the number of available CUDA devices
    print(f"Number of GPUs available: {torch.cuda.device_count()}")
else:
    print("No GPU available. Training will run on CPU.")


No GPU available. Training will run on CPU.


In [5]:
batch_size = 2
num_decoder_tokens = 3
num_encoder_tokens = 5
d_model = 8
num_heads = 2
head_dim = 4

In [6]:
w_q = nn.Linear(d_model, d_model)
w_k = nn.Linear(d_model, d_model)
w_v = nn.Linear(d_model, d_model)
w_o = nn.Linear(d_model, d_model)

In [24]:
dummy_encoder_input = torch.rand(batch_size, num_encoder_tokens, d_model)
dummy_decoder_input = torch.rand(batch_size, num_decoder_tokens, d_model)
print(dummy_encoder_input.shape)
print(dummy_decoder_input.shape)
print("(batch_size, num_tokens, embedding_dimension)")
dummy_decoder_input

torch.Size([2, 5, 8])
torch.Size([2, 3, 8])
(batch_size, num_tokens, embedding_dimension)


tensor([[[0.9637, 0.2981, 0.2920, 0.2988, 0.6512, 0.4938, 0.4864, 0.5161],
         [0.1392, 0.3185, 0.1296, 0.2811, 0.5358, 0.0804, 0.1570, 0.2260],
         [0.5709, 0.5854, 0.0758, 0.2455, 0.8940, 0.2138, 0.4794, 0.1917]],

        [[0.3716, 0.0680, 0.2557, 0.7858, 0.3653, 0.9351, 0.0470, 0.2177],
         [0.1854, 0.8090, 0.3264, 0.6470, 0.5951, 0.1694, 0.6307, 0.7659],
         [0.9204, 0.2156, 0.6745, 0.9588, 0.5823, 0.3682, 0.3088, 0.4921]]])

In [8]:
q = w_q(dummy_decoder_input)
k = w_k(dummy_encoder_input)
v = w_v(dummy_encoder_input)

In [12]:
print(q.shape)
print(k.shape)
print("(batch_size, num_tokens, key_dimension)")
q

torch.Size([2, 3, 8])
torch.Size([2, 5, 8])
(batch_size, num_tokens, key_dimension)


tensor([[[ 0.0789,  0.0434,  0.4044,  0.2277,  0.1883, -0.2614, -0.3559,
           0.3281],
         [ 0.0798,  0.1055,  0.3106,  0.3218,  0.1372, -0.2894, -0.3129,
           0.2878],
         [-0.0747,  0.0010,  0.3638,  0.1801,  0.2021, -0.4182, -0.1351,
           0.2753]],

        [[-0.1142,  0.4540,  0.5409,  0.2575,  0.2390, -0.2816, -0.4658,
           0.5887],
         [-0.1961,  0.2101,  0.3996,  0.2672,  0.0319, -0.3950,  0.0396,
           0.4506],
         [ 0.1618,  0.0488,  0.2134,  0.1144, -0.0071, -0.2386, -0.2005,
           0.2281]]], grad_fn=<ViewBackward0>)

In [13]:
q_head = q.view(batch_size, num_decoder_tokens, num_heads, head_dim)
k_head = k.view(batch_size, num_encoder_tokens, num_heads, head_dim)
v_head = v.view(batch_size, num_encoder_tokens, num_heads, head_dim)
print(q_head.shape)
print(k_head.shape)
q_head

torch.Size([2, 3, 2, 4])
torch.Size([2, 5, 2, 4])


tensor([[[[ 0.0789,  0.0434,  0.4044,  0.2277],
          [ 0.1883, -0.2614, -0.3559,  0.3281]],

         [[ 0.0798,  0.1055,  0.3106,  0.3218],
          [ 0.1372, -0.2894, -0.3129,  0.2878]],

         [[-0.0747,  0.0010,  0.3638,  0.1801],
          [ 0.2021, -0.4182, -0.1351,  0.2753]]],


        [[[-0.1142,  0.4540,  0.5409,  0.2575],
          [ 0.2390, -0.2816, -0.4658,  0.5887]],

         [[-0.1961,  0.2101,  0.3996,  0.2672],
          [ 0.0319, -0.3950,  0.0396,  0.4506]],

         [[ 0.1618,  0.0488,  0.2134,  0.1144],
          [-0.0071, -0.2386, -0.2005,  0.2281]]]], grad_fn=<ViewBackward0>)

In [16]:
q_head = q_head.transpose(1, 2)
k_head = k_head.transpose(1, 2)
v_head = v_head.transpose(1, 2)
print(q_head.shape)
print(k_head.shape)
print("(batch_size, num_heads, num_tokens, head_dim)")
q_head

torch.Size([2, 2, 3, 4])
torch.Size([2, 2, 5, 4])
(batch_size, num_heads, num_tokens, head_dim)


tensor([[[[ 0.0789,  0.0434,  0.4044,  0.2277],
          [ 0.0798,  0.1055,  0.3106,  0.3218],
          [-0.0747,  0.0010,  0.3638,  0.1801]],

         [[ 0.1883, -0.2614, -0.3559,  0.3281],
          [ 0.1372, -0.2894, -0.3129,  0.2878],
          [ 0.2021, -0.4182, -0.1351,  0.2753]]],


        [[[-0.1142,  0.4540,  0.5409,  0.2575],
          [-0.1961,  0.2101,  0.3996,  0.2672],
          [ 0.1618,  0.0488,  0.2134,  0.1144]],

         [[ 0.2390, -0.2816, -0.4658,  0.5887],
          [ 0.0319, -0.3950,  0.0396,  0.4506],
          [-0.0071, -0.2386, -0.2005,  0.2281]]]],
       grad_fn=<TransposeBackward0>)

In [17]:
k_t = k_head.transpose(-2, -1)
print(k_t.shape)
k_t

torch.Size([2, 2, 4, 5])


tensor([[[[ 0.1096, -0.1606,  0.1359,  0.5006,  0.1049],
          [ 0.3452,  0.5503,  0.4966,  0.4513,  0.4346],
          [ 0.2399,  0.1040,  0.0901,  0.5244, -0.0468],
          [ 0.4860,  0.3530,  0.0662,  0.2489,  0.0152]],

         [[ 0.1360, -0.0977,  0.1237,  0.4488,  0.0858],
          [-0.0651, -0.0720,  0.0515, -0.0421,  0.0549],
          [ 0.2814,  0.2503,  0.5525,  0.6914,  0.2700],
          [ 0.4662,  0.1553,  0.7384,  0.6299,  0.6219]]],


        [[[ 0.3108, -0.1881,  0.3779, -0.0564, -0.2032],
          [ 0.5797,  0.3780,  0.4065,  0.6812,  0.4742],
          [ 0.4512,  0.0488,  0.4045,  0.4023, -0.1135],
          [ 0.2764,  0.3504,  0.3914, -0.0117, -0.0244]],

         [[ 0.2539, -0.0666,  0.4356,  0.0151, -0.1215],
          [-0.0485, -0.0120, -0.1478,  0.2116,  0.1715],
          [ 0.7106,  0.0507,  0.5890,  0.4629,  0.0310],
          [ 0.4559,  0.2813,  0.5074,  0.5374,  0.5771]]]],
       grad_fn=<TransposeBackward0>)

In [18]:
sim1 = (q_head @ k_t)/math.sqrt(head_dim)
print(sim1.shape)
print("(batch, num_heads, num_queries, num_keys)")
sim1
# Attention Scores Matrix

torch.Size([2, 2, 3, 5])
(batch, num_heads, num_queries, num_keys)


tensor([[[[ 0.1156,  0.0668,  0.0419,  0.1639,  0.0058],
          [ 0.1380,  0.0956,  0.0562,  0.1652,  0.0223],
          [ 0.0835,  0.0570,  0.0175,  0.0993, -0.0109]],

         [[ 0.0477, -0.0189,  0.0277,  0.0281,  0.0549],
          [ 0.0418, -0.0131,  0.0208,  0.0193,  0.0452],
          [ 0.0725,  0.0097,  0.0660,  0.0941,  0.0646]]],


        [[[ 0.2715,  0.1548,  0.2305,  0.2651,  0.0854],
          [ 0.1575,  0.1147,  0.1388,  0.1559,  0.0438],
          [ 0.1032,  0.0192,  0.1060,  0.0543, -0.0184]],

         [[ 0.0059,  0.0647,  0.0850,  0.0224,  0.1240],
          [ 0.1304,  0.0657,  0.1621,  0.0887,  0.0948],
          [-0.0144,  0.0287,  0.0149, -0.0104,  0.0427]]]],
       grad_fn=<DivBackward0>)

In [25]:
sim2 = F.softmax(sim1, dim=-1)
print(sim2.shape)
print("(num_batches, num_heads, num_queries, num_keys)")
sim2
# Attention Weights matrix
# Basically building a neural network for value vectors

torch.Size([2, 2, 3, 5])
(num_batches, num_heads, num_queries, num_keys)


tensor([[[[0.2072, 0.1973, 0.1925, 0.2174, 0.1856],
          [0.2084, 0.1997, 0.1920, 0.2142, 0.1856],
          [0.2068, 0.2014, 0.1936, 0.2101, 0.1882]],

         [[0.2039, 0.1908, 0.1999, 0.2000, 0.2054],
          [0.2038, 0.1929, 0.1996, 0.1993, 0.2045],
          [0.2022, 0.1898, 0.2009, 0.2066, 0.2006]]],


        [[[0.2140, 0.1904, 0.2054, 0.2126, 0.1776],
          [0.2070, 0.1983, 0.2032, 0.2067, 0.1848],
          [0.2101, 0.1932, 0.2107, 0.2001, 0.1860]],

         [[0.1892, 0.2007, 0.2048, 0.1924, 0.2129],
          [0.2043, 0.1915, 0.2109, 0.1960, 0.1972],
          [0.1947, 0.2033, 0.2005, 0.1955, 0.2061]]]],
       grad_fn=<SoftmaxBackward0>)

In [26]:
print(v_head.shape)
v_head # Value vectors

torch.Size([2, 2, 5, 4])


tensor([[[[-2.3781e-02, -1.0877e-01, -1.8671e-01,  2.4986e-01],
          [-1.9516e-01, -1.1030e-01,  1.7358e-03,  2.7807e-01],
          [ 5.8743e-02,  3.8024e-01, -1.7357e-01,  3.3038e-01],
          [-2.6916e-01,  3.4614e-01, -2.3440e-01,  2.0566e-01],
          [-3.2097e-02,  2.5604e-01, -1.2853e-01,  4.4076e-01]],

         [[-6.2408e-01, -1.5500e-02,  1.7296e-02, -8.1169e-01],
          [-8.8053e-01,  1.8640e-01,  3.7726e-01, -6.8672e-01],
          [-4.4445e-01,  4.6805e-01,  6.8170e-02, -9.6123e-01],
          [-6.4067e-01,  2.0585e-02,  7.8225e-02, -7.5566e-01],
          [-3.7588e-01,  3.9066e-01,  1.2917e-01, -8.5113e-01]]],


        [[[-2.6472e-01,  3.3983e-01, -1.6541e-01,  1.1485e-01],
          [ 5.5485e-02, -2.7136e-01, -1.5433e-01,  4.6708e-01],
          [-3.3152e-01,  1.5291e-01, -8.1214e-03,  1.0648e-01],
          [-2.6710e-02,  1.9233e-01, -2.0188e-01,  5.2181e-01],
          [ 1.2331e-01,  2.0802e-02, -9.7182e-02,  6.1993e-01]],

         [[-7.0838e-01,  1.0592e

In [32]:
sim3 = sim2 @ v_head
print(sim3.shape)
sim3 
# Applying Attention Weights to value vectors
# Essentially, passing value vectors through a neural network

torch.Size([2, 2, 3, 4])


tensor([[[[-0.0966,  0.1517, -0.1466,  0.2968],
          [-0.0963,  0.1500, -0.1460,  0.2969],
          [-0.0954,  0.1498, -0.1453,  0.2978]],

         [[-0.5894,  0.2103,  0.1313, -0.8146],
          [-0.5903,  0.2102,  0.1319, -0.8143],
          [-0.5903,  0.2089,  0.1309, -0.8143]]],


        [[[-0.0979,  0.0970, -0.1266,  0.3564],
          [-0.0939,  0.0912, -0.1262,  0.3605],
          [-0.0971,  0.0935, -0.1247,  0.3565]],

         [[-0.6581,  0.1719,  0.1069, -0.7857],
          [-0.6626,  0.1684,  0.1107, -0.7818],
          [-0.6603,  0.1701,  0.1063, -0.7861]]]],
       grad_fn=<UnsafeViewBackward0>)

In [33]:
sim3 = sim3.transpose(1, 2).contiguous()
sim3.shape

torch.Size([2, 3, 2, 4])

In [34]:
sim3 = sim3.view(batch_size, num_decoder_tokens, d_model)
print(sim3.shape)
sim3

torch.Size([2, 3, 8])


tensor([[[-0.0966,  0.1517, -0.1466,  0.2968, -0.5894,  0.2103,  0.1313,
          -0.8146],
         [-0.0963,  0.1500, -0.1460,  0.2969, -0.5903,  0.2102,  0.1319,
          -0.8143],
         [-0.0954,  0.1498, -0.1453,  0.2978, -0.5903,  0.2089,  0.1309,
          -0.8143]],

        [[-0.0979,  0.0970, -0.1266,  0.3564, -0.6581,  0.1719,  0.1069,
          -0.7857],
         [-0.0939,  0.0912, -0.1262,  0.3605, -0.6626,  0.1684,  0.1107,
          -0.7818],
         [-0.0971,  0.0935, -0.1247,  0.3565, -0.6603,  0.1701,  0.1063,
          -0.7861]]], grad_fn=<ViewBackward0>)

In [23]:
output = w_o(sim3)
# Projecting from key_dimension back to d_model (up projection)
print(output.shape)
output
# We have output tensor of the same dimension as our decoder input. Ready to be passed on to the next layer.

torch.Size([2, 3, 8])


tensor([[[-0.5583,  0.0560, -0.0584, -0.1336,  0.0418,  0.1040, -0.1500,
          -0.1611],
         [-0.5579,  0.0563, -0.0584, -0.1333,  0.0422,  0.1033, -0.1501,
          -0.1609],
         [-0.5578,  0.0560, -0.0589, -0.1332,  0.0420,  0.1025, -0.1498,
          -0.1606]],

        [[-0.5486,  0.0621, -0.0669, -0.1215,  0.0435,  0.0471, -0.1618,
          -0.1180],
         [-0.5455,  0.0631, -0.0664, -0.1205,  0.0439,  0.0435, -0.1618,
          -0.1145],
         [-0.5481,  0.0621, -0.0671, -0.1210,  0.0436,  0.0449, -0.1621,
          -0.1175]]], grad_fn=<ViewBackward0>)

In [ ]:
class CrossMultiHeadAttention(nn.Module):
    """
    Compute Scaled Dot Product Attention using Multiple Heads
    """

    def __init__(self, d_model:int, num_heads: int):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.w_q = nn.Linear(self.d_model, self.d_model)
        self.w_k = nn.Linear(self.d_model, self.d_model)
        self.w_v = nn.Linear(self.d_model, self.d_model)
        self.w_o = nn.Linear(self.d_model, self.d_model)

        # Checking if d_model is divisible by num_heads
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        self.head_dim = d_model // num_heads # Dimension of query, key and value vectors within each head

    def forward(self, encoder_input: torch.Tensor, decoder_input: torch.Tensor) -> torch.Tensor:
        
        # Checking if input has correct embedding dimension
        if encoder_input.shape[-1] != self.d_model:
            raise ValueError(f"Expected encoder input feature dimension to be d_model ({self.d_model}), but got {encoder_input.shape[-1]}")

        if decoder_input.shape[-1] != self.d_model:
            raise ValueError(f"Expected decoder input feature dimension to be d_model ({self.d_model}), but got {decoder_input.shape[-1]}")

        # Input Dimension: (batch_size, num_tokens, d_model)
        batch_size = decoder_input.shape[0]
        encoder_num_tokens = encoder_input.shape[1]
        decoder_num_tokens = decoder_input.shape[1]

        # Creating query, key and value vectors from input embeddings. 
        # These vectors contain queries from all the heads combined.
        q = self.w_q(decoder_input)
        k = self.w_k(encoder_input)
        v = self.w_v(encoder_input)
        # Dimension: (batch_sizee, num_tokens, d_model)

        # Splitting our vectors into heads by reshaping our tensors.
        q_head = q.view(batch_size, decoder_num_tokens, self.num_heads, self.head_dim)
        k_head = k.view(batch_size, encoder_num_tokens, self.num_heads, self.head_dim)
        v_head = v.view(batch_size, encoder_num_tokens, self.num_heads, self.head_dim)
        # Dimension: (batch_size, num_tokens, num_heads, head_dimension)

        # Swapping (num_tokens) and (num_heads) dimensions.
        # We do this because we need (num_tokens) and (head_dimension) at the end.
        # This is because we need to perform matrix operations on the last two dimensions
        q_head = q_head.transpose(1, 2)
        k_head = k_head.transpose(1, 2)
        v_head = v_head.transpose(1, 2)
        # Dimension: (batch_size, num_heads, num_tokens, head_dimension)

        # Getting a transpose of key vectors for matrix multiplication 
        k_t = k_head.transpose(-2, -1)
        sim1 = (q_head @ k_t)/math.sqrt(self.head_dim) # Scaled dot product of query and key vectors
        # sim1 is the matrix of attention scores for each token
        # Dimension: (batch_size, num_heads, num_queries, num_keys)

        # Performing softmax along the keys dimension to get attention scores for each key
        sim2 = F.softmax(sim1, dim=-1)
        # Dimension: (batch_size, num_heads, num_queries, num_keys)

        # Weighted sum of attention weights and value vectors
        sim3 = sim2 @ v_head
        # Dimension: (batch_size, num_heads, num_queries, head_dimension)

        # Swapping (num_heads) and (num_queries) dimension back so that we can recombine all the heads
        sim3 = sim3.transpose(1, 2).contiguous()

        # Recombining all the heads
        sim3 = sim3.view(batch_size, decoder_num_tokens, self.d_model)
        # Dimension: (batch_size, num_tokens, d_model)
        # Same as input dimension
        
        # Passing all of our heads into a final layer so that all heads can interact.
        output = self.w_o(sim3)
        # Dimension: (batch_size, num_tokens, d_model)
        # Same as input dimension
        
        return output

In [36]:
dummy_input = torch.rand(2, 3, 8)
dummy_input

tensor([[[0.8542, 0.4894, 0.1183, 0.6878, 0.7106, 0.9473, 0.4460, 0.5657],
         [0.3933, 0.3094, 0.3904, 0.7526, 0.7817, 0.1441, 0.3017, 0.2920],
         [0.7340, 0.2864, 0.2676, 0.3791, 0.3603, 0.1579, 0.4686, 0.5569]],

        [[0.8153, 0.1535, 0.5322, 0.1786, 0.5677, 0.1548, 0.9260, 0.1895],
         [0.6205, 0.6270, 0.5606, 0.6078, 0.5168, 0.1525, 0.0170, 0.8796],
         [0.9641, 0.9985, 0.7492, 0.8134, 0.1294, 0.7502, 0.8513, 0.8271]]])

In [37]:
mha_layer = CrossMultiHeadAttention(d_model=8, num_heads=2)

In [40]:
output = mha_layer(dummy_encoder_input, dummy_decoder_input)
output

tensor([[[-0.1865,  0.5094,  0.3435, -0.0368, -0.4556, -0.3861,  0.1135,
          -0.4207],
         [-0.1890,  0.5122,  0.3451, -0.0387, -0.4568, -0.3843,  0.1068,
          -0.4257],
         [-0.1881,  0.5112,  0.3452, -0.0380, -0.4569, -0.3847,  0.1090,
          -0.4236]],

        [[-0.1314,  0.4716,  0.2015, -0.0570, -0.3957, -0.4300,  0.1421,
          -0.4395],
         [-0.1308,  0.4715,  0.2011, -0.0571, -0.3974, -0.4323,  0.1423,
          -0.4401],
         [-0.1320,  0.4722,  0.2013, -0.0578, -0.3941, -0.4263,  0.1397,
          -0.4396]]], grad_fn=<ViewBackward0>)